# ETL — Lending Club → Neon Postgres (Dimensional Model)

Loads resolved loans, builds a star schema (fact_loans + dimensions),
and writes to Neon. Reads credentials from ../.env (gitignored).


In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

DATA_PATH = "../data/raw/loan.csv"

cols_to_load = [
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "issue_d", "purpose", "addr_state",
    "dti", "delinq_2yrs", "earliest_cr_line",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "mort_acc", "pub_rec_bankruptcies",
    "loan_status",
]

df = pd.read_csv(DATA_PATH, usecols=cols_to_load, low_memory=False)
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["default"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)
print("Working set:", df.shape)

Working set: (1303638, 26)


In [2]:
load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).fetchone()[0])

PostgreSQL 18.4 (6e15e70) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [3]:
schema_sql = """
DROP TABLE IF EXISTS fact_loans CASCADE;
DROP TABLE IF EXISTS dim_grade CASCADE;
DROP TABLE IF EXISTS dim_borrower CASCADE;
DROP TABLE IF EXISTS dim_loan_terms CASCADE;
DROP TABLE IF EXISTS dim_date CASCADE;
DROP TABLE IF EXISTS dim_geography CASCADE;

CREATE TABLE dim_grade (
    grade_key      SERIAL PRIMARY KEY,
    grade          VARCHAR(2)  NOT NULL,
    sub_grade      VARCHAR(3)  NOT NULL,
    UNIQUE (grade, sub_grade)
);

CREATE TABLE dim_borrower (
    borrower_key         SERIAL PRIMARY KEY,
    home_ownership       VARCHAR(20),
    emp_length           VARCHAR(20),
    verification_status  VARCHAR(20),
    UNIQUE (home_ownership, emp_length, verification_status)
);

CREATE TABLE dim_loan_terms (
    terms_key   SERIAL PRIMARY KEY,
    term        VARCHAR(15),
    purpose     VARCHAR(30),
    UNIQUE (term, purpose)
);

CREATE TABLE dim_date (
    date_key    SERIAL PRIMARY KEY,
    issue_d     VARCHAR(10),
    issue_year  INT,
    issue_month INT,
    issue_quarter INT,
    UNIQUE (issue_d)
);

CREATE TABLE dim_geography (
    geo_key     SERIAL PRIMARY KEY,
    addr_state  VARCHAR(2),
    UNIQUE (addr_state)
);

CREATE TABLE fact_loans (
    loan_id      BIGSERIAL PRIMARY KEY,
    grade_key    INT REFERENCES dim_grade(grade_key),
    borrower_key INT REFERENCES dim_borrower(borrower_key),
    terms_key    INT REFERENCES dim_loan_terms(terms_key),
    date_key     INT REFERENCES dim_date(date_key),
    geo_key      INT REFERENCES dim_geography(geo_key),
    loan_amnt    NUMERIC(12,2),
    int_rate     NUMERIC(6,3),
    installment  NUMERIC(10,2),
    annual_inc   NUMERIC(14,2),
    dti          NUMERIC(7,2),
    revol_util   NUMERIC(7,2),
    default_flag INT NOT NULL
);
"""

with engine.begin() as conn:
    for statement in schema_sql.strip().split(";"):
        if statement.strip():
            conn.execute(text(statement))

print("Schema created — 5 dimensions + fact_loans")

Schema created — 5 dimensions + fact_loans


In [4]:
import numpy as np

etl = df.copy()

# Clean numeric-ish text fields
etl["term"] = etl["term"].str.strip()  # "36 months" / "60 months"
etl["revol_util"] = (etl["revol_util"]
                     .astype(str).str.replace("%", "", regex=False))
etl["revol_util"] = pd.to_numeric(etl["revol_util"], errors="coerce")

# Parse issue_d (format like "Dec-2015") into date parts
etl["issue_dt"] = pd.to_datetime(etl["issue_d"], format="%b-%Y", errors="coerce")
etl["issue_year"] = etl["issue_dt"].dt.year
etl["issue_month"] = etl["issue_dt"].dt.month
etl["issue_quarter"] = etl["issue_dt"].dt.quarter

# Quick data-quality check on key fields
print("Null counts on key fields:")
print(etl[["term", "revol_util", "issue_dt", "annual_inc", "dti"]].isnull().sum())
print("\nRows:", len(etl))

Null counts on key fields:
term            0
revol_util    810
issue_dt        0
annual_inc      0
dti           312
dtype: int64

Rows: 1303638


In [5]:
# Confirm dimension cardinalities are low (sanity before building dims)
for col in ["grade", "sub_grade", "home_ownership", "emp_length",
            "verification_status", "term", "purpose", "addr_state"]:
    print(f"{col}: {etl[col].nunique()} unique")

grade: 7 unique
sub_grade: 35 unique
home_ownership: 6 unique
emp_length: 11 unique
verification_status: 3 unique
term: 2 unique
purpose: 14 unique
addr_state: 51 unique


In [6]:
# Build dimension tables from unique combinations, load to Neon, read back keys

def load_dimension(table, cols):
    """Write unique rows of `cols` to `table`, return df with surrogate keys."""
    dim = etl[cols].drop_duplicates().reset_index(drop=True)
    dim.to_sql(table, engine, if_exists="append", index=False)
    # Read back with the generated surrogate keys
    return pd.read_sql(f"SELECT * FROM {table}", engine)

dim_grade    = load_dimension("dim_grade", ["grade", "sub_grade"])
dim_borrower = load_dimension("dim_borrower",
                  ["home_ownership", "emp_length", "verification_status"])
dim_terms    = load_dimension("dim_loan_terms", ["term", "purpose"])
dim_geo      = load_dimension("dim_geography", ["addr_state"])

# dim_date needs the derived parts too
dim_date_src = (etl[["issue_d", "issue_year", "issue_month", "issue_quarter"]]
                .drop_duplicates().reset_index(drop=True))
dim_date_src.to_sql("dim_date", engine, if_exists="append", index=False)
dim_date = pd.read_sql("SELECT * FROM dim_date", engine)

print("dim_grade:",    len(dim_grade))
print("dim_borrower:", len(dim_borrower))
print("dim_terms:",    len(dim_terms))
print("dim_geo:",      len(dim_geo))
print("dim_date:",     len(dim_date))

dim_grade: 35
dim_borrower: 192
dim_terms: 28
dim_geo: 51
dim_date: 139


In [7]:
# Merge surrogate keys back onto the main dataset
fact = etl.merge(dim_grade,    on=["grade", "sub_grade"], how="left") \
          .merge(dim_borrower, on=["home_ownership", "emp_length", "verification_status"], how="left") \
          .merge(dim_terms,    on=["term", "purpose"], how="left") \
          .merge(dim_geo,      on=["addr_state"], how="left") \
          .merge(dim_date[["date_key", "issue_d"]], on="issue_d", how="left")

# Select only the fact-table columns
fact_load = fact[[
    "grade_key", "borrower_key", "terms_key", "date_key", "geo_key",
    "loan_amnt", "int_rate", "installment", "annual_inc",
    "dti", "revol_util", "default"
]].rename(columns={"default": "default_flag"})

# Validation: no missing foreign keys before load
print("Null FK check (should all be 0):")
print(fact_load[["grade_key","borrower_key","terms_key","date_key","geo_key"]].isnull().sum())
print("\nRows to load:", len(fact_load))

Null FK check (should all be 0):
grade_key       0
borrower_key    0
terms_key       0
date_key        0
geo_key         0
dtype: int64

Rows to load: 1303638


In [8]:
import time

start = time.time()

fact_load.to_sql(
    "fact_loans",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000,    # send in batches of 10k rows
    method="multi"      # multi-row INSERT — much faster over a network
)

print(f"Loaded {len(fact_load):,} rows in {time.time()-start:.0f}s")

Loaded 1,303,638 rows in 229s


In [9]:
query = """
SELECT
    g.grade,
    COUNT(*)                              AS total_loans,
    ROUND(AVG(f.default_flag) * 100, 1)   AS default_rate_pct
FROM fact_loans f
JOIN dim_grade g ON f.grade_key = g.grade_key
GROUP BY g.grade
ORDER BY g.grade;
"""

pd.read_sql(query, engine)

,grade,total_loans,default_rate_pct
0,A,226245,6.1
1,B,380158,13.4
2,C,369937,22.5
3,D,195288,30.4
4,E,91574,38.6
5,F,31485,45.3
6,G,8951,50.1
